## Validación del aprendizaje centralizado

Este notebook implementa el escenario base de entrenamiento centralizado utilizado como referencia para el resto de experimentos del proyecto. Sobre el dataset CIFAR-10 se entrena una red convolucional simple (SimpleCNN) utilizando la totalidad de los datos en un único nodo, sin aplicar técnicas de aprendizaje federado.

Además del entrenamiento y evaluación del modelo, se integra CodeCarbon para capturar métricas de sostenibilidad, incluyendo tiempo de ejecución, consumo energético y emisiones operativas de carbono. Al finalizar el experimento, los resultados de rendimiento y las métricas ambientales se almacenan en un fichero CSV estructurado, que servirá como punto de comparación para los posteriores escenarios federados y para el cálculo futuro del Software Carbon Intensity (SCI).

0. Constantes de definición del entorno

In [1]:
DATA_DIR = "../data"
PROBES_DIR = "../probes"

DATASET = "CIFAR10"
SCENARIO = "centralized"
MODEL_NAME = "SimpleCNN"
BATCH_SIZE = 64
EPOCHS = 2
LEARNING_RATE = 0.001
RANDOM_SEED = 42

## Temporal

DEVICE_CATEGORY = "laptop"
DEVICE_YEAR = 2023

1. Carga del Dataset y DataLoader

In [2]:

import torch

from torch.utils.data import DataLoader

from torchvision import datasets
from torchvision import transforms

# ==================================================
# Transformaciones
# ==================================================

transform = transforms.ToTensor()

# ==================================================
# Dataset
# ==================================================

train_dataset = datasets.CIFAR10(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform
)

# ==================================================
# DataLoader
# ==================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


2. Modelo

In [3]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

3. Modelo local para pruebas

In [4]:
model = SimpleCNN()

print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


4. Función de pérdida y optimización

In [5]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

5. Función de entrenamiento

In [6]:
def train_model(
    model,
    train_loader,
    optimizer,
    criterion,
    epochs
):

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0

        for images, labels in train_loader:

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Loss: {epoch_loss:.4f}"
        )

6. Función de evaluación

In [7]:
def evaluate_model(
    model,
    test_loader
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = correct / total

    return accuracy

7. Función de entrenamiento y captura de datos de emisiones

In [ ]:

from codecarbon import EmissionsTracker
from pathlib import Path

def run_training_with_codecarbon(
    train_function,
    probe_id,
    scenario,
    client_id=1,
    output_dir=PROBES_DIR
):

    Path(output_dir).mkdir(
        parents=True,
        exist_ok=True
    )

    tracker = EmissionsTracker(
        output_dir=output_dir,
        output_file=(
            f"cc_{probe_id}_{scenario}_{client_id}.csv"
        )
    )

    tracker.start()

    tracker.start_task("training")

    train_function()

    training_data = tracker.stop_task()

    tracker.stop()

# Eliminar archivos de emisiones base 
    for file in Path(output_dir).glob("emissions_base*.csv"):
        file.unlink()

    return {
        "client_id": client_id,
        "energy_total_kwh": training_data.energy_consumed,
        "emissions_total_kg": training_data.emissions,
        "training_time_total_s": training_data.duration
}

8. Función de ejecución del experimento
y captura de datos del probe (code carbon (cc) y entorno (env))

In [9]:
import time
from pathlib import Path
import pandas as pd
import uuid


def run_experiment(
    scenario,
    model,
    train_loader,
    test_loader,
    optimizer,
    criterion,
    epochs
):

    probe_id = str(uuid.uuid4())

    cc_metrics = run_training_with_codecarbon(
    train_function=lambda: train_model(
        model,
        train_loader,
        optimizer,
        criterion,
        EPOCHS
    ),
    probe_id=probe_id,
    scenario=SCENARIO
)

    accuracy = evaluate_model(
        model,
        test_loader
    )

    env = {

        # ==================================================
        # Identificación
        # ==================================================

        "probe_id": probe_id,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),

        # ==================================================
        # Dispositivo
        # ==================================================

        "device_category": DEVICE_CATEGORY,
        "device_year": DEVICE_YEAR,

        # ==================================================
        # Escenario experimental
        # ==================================================

        "scenario": SCENARIO,
        "dataset": DATASET,
        "model": MODEL_NAME,

        # ==================================================
        # Topología FL
        # ==================================================

        "num_clients": 1,
        "num_rounds": 1,

        # ==================================================
        # Configuración de entrenamiento
        # ==================================================

        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,

        # ==================================================
        # Tamaño del trabajo
        # ==================================================

        "training_samples": len(train_loader.dataset),

        # ==================================================
        # Resultados
        # ==================================================

        "accuracy": accuracy,
        "energy_total_kwh": cc_metrics["energy_total_kwh"],
        "emissions_total_kg": cc_metrics["emissions_total_kg"],
        "training_time_total_s": cc_metrics["training_time_total_s"]
    }

    probes_file = (
        f"{PROBES_DIR}/env_{probe_id}_{scenario}.csv"
    )

    pd.DataFrame([env]).to_csv(
        probes_file,
        index=False
    )

Ejecución del experimento:

In [10]:
import pandas as pd
from pathlib import Path

run_experiment(
    scenario=SCENARIO,
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS
)


[codecarbon WARNING @ 12:08:45] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 12:08:45] Emissions data (if any) will be saved to file /home/rutpal/TFM/probes/cc_9560988a-3251-44e9-a608-f705cdce5fa6_centralized_client_1.csv
[codecarbon INFO @ 12:08:45] [setup] RAM Tracking...
[codecarbon INFO @ 12:08:45] [setup] CPU Tracking...
[codecarbon INFO @ 12:08:45] [setup] GPU Tracking...
[codecarbon INFO @ 12:08:45] No GPU found.
[codecarbon WARNING @ 12:08:47] We saw that you have a 12th Gen Intel(R) Core(TM) i7-1255U but we don't know it. Please contact us.
[codecarbon WARNING @ 12:08:47] We will use the default power consumption of 4 W per thread for your 12 CPU, so 48W.
[codecarbon WARNING @ 12:08:47] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 12:08:47] CPU Model on c

Epoch 1/2 - Loss: 1.4786


[codecarbon INFO @ 12:09:49] Energy consumed for RAM : 0.000164 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 12:09:49] Delta energy consumed for CPU with cpu_load : 0.000175 kWh, power : 10.699912915118645 W
[codecarbon INFO @ 12:09:49] Energy consumed for All CPU : 0.000175 kWh
[codecarbon INFO @ 12:09:49] 0.000338 kWh of electricity and 0.000000 L of water were used since the beginning.


Epoch 2/2 - Loss: 1.1227
